In [14]:
import os, glob
import numpy as np
import tensorflow as tf

## Prepare dataset

In [15]:
CAPTURES = "../dataset/captures"
MAX_LEN  = 100 
N_CH = 6 
classes = sorted(d for d in os.listdir(CAPTURES)
                 if os.path.isdir(os.path.join(CAPTURES, d)))

class_to_id = {c: i for i, c in enumerate(classes)}

# Transform the 2D dimension Matrix into a 1D vector
X, y = [], []
for cls in classes:
    for path in glob.glob(os.path.join(CAPTURES, cls, "*.csv")):
        rows = np.loadtxt(path, delimiter=",", skiprows=1, usecols=range(4, 10))
        seq = np.zeros((MAX_LEN, N_CH), dtype=np.float32)   
        n = min(rows.shape[0], MAX_LEN)
        seq[:n] = rows[:n]
        X.append(seq.flatten())          
        y.append(class_to_id[cls])

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.int32)
print("X", X.shape, "classes", len(classes))  


X (492, 600) classes 16


In [19]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)   # stratify: 16 classes

scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train).astype(np.float32)
X_test  = scaler.transform(X_test).astype(np.float32)


model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(N_CH * MAX_LEN,)),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(64,  activation="relu"),
    tf.keras.layers.Dense(len(classes), activation="softmax"),
])
model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

model.fit(X_train, y_train, batch_size=32, epochs=100, validation_data=(X_test, y_test))

Epoch 1/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.4809 - loss: 1.9846 - val_accuracy: 0.7374 - val_loss: 1.2031
Epoch 2/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7888 - loss: 0.9316 - val_accuracy: 0.8687 - val_loss: 0.6951
Epoch 3/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8677 - loss: 0.5571 - val_accuracy: 0.9596 - val_loss: 0.4350
Epoch 4/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9135 - loss: 0.3590 - val_accuracy: 0.9293 - val_loss: 0.3368
Epoch 5/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9262 - loss: 0.2680 - val_accuracy: 0.9192 - val_loss: 0.3077
Epoch 6/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9389 - loss: 0.2188 - val_accuracy: 0.9192 - val_loss: 0.3007
Epoch 7/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9466 - loss: 0.1837 - val_accuracy: 0.9192 - val_loss: 0.2972
Epoch 8/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9491 - loss: 0.1574 - val_accuracy: 0.9192 - 

In [ ]:
def evaluate(model, X_test, y_test, class_names):
    """Evaluate a trained classifier: accuracy, per-class report, top confusions."""
    y_pred = np.argmax(model.predict(X_test), axis=1)
    acc = accuracy_score(y_test, y_pred)

    print(f'Test accuracy: {acc:.4f}')
    print()
    print(classification_report(y_test, y_pred, target_names=class_names))

    cm = confusion_matrix(y_test, y_pred)
    mistakes = []
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            if i != j and cm[i, j]:
                mistakes.append((cm[i, j], class_names[i], class_names[j]))
    mistakes.sort(reverse=True)
    print('Most common confusions (true -> predicted):')
    for n, t, p in mistakes[:10]:
        print(f'  {t} -> {p}: {n}')
    return acc, y_pred, cm